# 🚀 MLOps CIA1 — Online Shoppers Purchasing Intention

**Student:** Shubham Patil  
**Dataset:** UCI Online Shoppers Purchasing Intention Dataset  
**GitHub:** [shubham80patil/MLOps-CIA1](https://github.com/shubham80patil/MLOps-CIA1)  

---

## Assignment Tasks

| Task | Description | Tools |
|------|-------------|-------|
| **Task 1** | MLOps Lifecycle Analysis | Diagram + Code |
| **Task 2** | Experiment Tracking | MLflow |
| **Task 3** | Dataset Versioning | DVC + Git |
| **Task 4** | Workflow Automation | Kubeflow (Simulated) |
| **Task 5** | Feature Store & Cloud Study | Feast + Vertex AI |

## 📦 Environment Setup

In [ ]:
# ----- Environment detection -----
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab — installing dependencies...")
    !pip install -q mlflow scikit-learn xgboost matplotlib seaborn pandas numpy joblib
    # Clone repo in Colab
    if not os.path.exists('MLOps-CIA1'):
        !git clone https://github.com/shubham80patil/MLOps-CIA1.git
    os.chdir('MLOps-CIA1')
    print("Working directory:", os.getcwd())
else:
    print("Running locally")
    print("Working directory:", os.getcwd())

In [ ]:
# ----- Imports -----
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
import xgboost as xgb

# MLflow
import mlflow
import mlflow.sklearn

import joblib
import json
from datetime import datetime

print("All imports successful")
print(f"pandas={pd.__version__}, sklearn={__import__('sklearn').__version__}, mlflow={mlflow.__version__}")

---

# Task 1: MLOps Lifecycle Analysis (4 Marks)

## End-to-End ML Lifecycle Diagram

```
+---------------------------------------------------------------------+
|                    ML LIFECYCLE - Online Shoppers                    |
+---------------------------------------------------------------------+
|                                                                     |
|  +--------------+    +------------------+    +------------------+   |
|  |  1. DATA      |--->  2. DATA          |--->  3. MODEL        |   |
|  |  INGESTION    |    |  PREPROCESSING    |    |  TRAINING        |   |
|  |              |    |                  |    |                  |   |
|  | - Load CSV   |    | - Handle NAs     |    | - Logistic Reg   |   |
|  | - Validate   |    | - Encode cats    |    | - Random Forest  |   |
|  | - DVC track  |    | - Scale features |    | - XGBoost        |   |
|  |              |    | - Feature eng.   |    | - Grad. Boost    |   |
|  +--------------+    +------------------+    +--------+---------+   |
|                                                        |            |
|                                                        v            |
|  +--------------+    +------------------+    +------------------+   |
|  |  6. MONITOR   |<---|  5. DEPLOYMENT    |<---|  4. MODEL        |   |
|  |              |    |                  |    |  EVALUATION      |   |
|  | - Pred drift |    | - Vertex AI      |    |                  |   |
|  | - Data drift |    | - REST API       |    | - Accuracy       |   |
|  | - Perf decay |    | - Model serving  |    | - Precision      |   |
|  | - Errors     |    | - CI/CD          |    | - Recall / F1    |   |
|  +------+-------+    +------------------+    | - ROC-AUC        |   |
|         |                                    | - MLflow compare |   |
|         +------------ Retrain Loop --------->+------------------+   |
|                                                                     |
|  Tools: MLflow | DVC | Kubeflow | Feast | Google Vertex AI          |
+---------------------------------------------------------------------+
```

### Stage Descriptions

| Stage | Description | Key Activities |
|-------|-------------|----------------|
| **1. Data Ingestion** | Load raw data and validate schema | Read CSV, check columns, DVC versioning |
| **2. Data Preprocessing** | Clean and transform data | Handle missing values, encode categoricals, scale features |
| **3. Model Training** | Train multiple ML models | LogReg, RF, XGBoost, GBM with MLflow tracking |
| **4. Model Evaluation** | Compare model performance | Accuracy, Precision, Recall, F1, ROC-AUC |
| **5. Deployment** | Deploy best model | Google Vertex AI endpoint, REST API |
| **6. Monitoring** | Monitor in production | Prediction drift, data drift, performance decay, errors |

### Stage 1: Data Ingestion

In [ ]:
# ======================== STAGE 1: DATA INGESTION ========================
DATA_PATH = "Dataset/online_shoppers_intention.csv"

df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print(f"\nColumn names:\n{df_raw.columns.tolist()}")
print(f"\nData types:\n{df_raw.dtypes}")
df_raw.head()

In [ ]:
# ----- Data Validation -----
print("=" * 60)
print("DATA VALIDATION REPORT")
print("=" * 60)

print(f"\nShape: {df_raw.shape}")
print(f"\nMissing values per column:")
missing = df_raw.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "   No missing values found")

print(f"\nDuplicate rows: {df_raw.duplicated().sum()}")

print(f"\nTarget distribution (Revenue):")
rev_counts = df_raw['Revenue'].value_counts()
print(f"   False (No purchase): {rev_counts[False]} ({rev_counts[False]/len(df_raw)*100:.1f}%)")
print(f"   True  (Purchase):    {rev_counts[True]}  ({rev_counts[True]/len(df_raw)*100:.1f}%)")
print(f"   Imbalance ratio:     {rev_counts[False]/rev_counts[True]:.2f}:1")

In [ ]:
# ----- Exploratory Data Analysis -----
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploratory Data Analysis - Online Shoppers Intention', fontsize=16, fontweight='bold')

# 1. Target distribution
colors = ['#2196F3', '#FF5722']
df_raw['Revenue'].value_counts().plot.pie(
    ax=axes[0, 0], autopct='%1.1f%%', colors=colors,
    labels=['No Purchase', 'Purchase'], startangle=90
)
axes[0, 0].set_title('Target Distribution (Revenue)', fontweight='bold')
axes[0, 0].set_ylabel('')

# 2. Monthly visitor distribution
month_order = ['Feb', 'Mar', 'May', 'June', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_data = df_raw['Month'].value_counts().reindex([m for m in month_order if m in df_raw['Month'].values])
month_data.plot.bar(ax=axes[0, 1], color='#4CAF50', edgecolor='white')
axes[0, 1].set_title('Sessions by Month', fontweight='bold')
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Count')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Visitor type distribution
df_raw['VisitorType'].value_counts().plot.bar(ax=axes[1, 0], color=['#9C27B0', '#FF9800', '#607D8B'], edgecolor='white')
axes[1, 0].set_title('Visitor Type Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Visitor Type')
axes[1, 0].set_ylabel('Count')
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. PageValues by Revenue
df_raw.boxplot(column='PageValues', by='Revenue', ax=axes[1, 1])
axes[1, 1].set_title('PageValues by Revenue', fontweight='bold')
axes[1, 1].set_xlabel('Revenue (Purchase?)')
axes[1, 1].set_ylabel('PageValues')
plt.suptitle('')  # Remove automatic title

plt.tight_layout()
os.makedirs('models', exist_ok=True)
plt.savefig('models/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("EDA plots saved")

### Stage 2: Data Preprocessing

In [ ]:
# ======================== STAGE 2: DATA PREPROCESSING ========================
df = df_raw.copy()

# --- Handle missing values ---
print(f"Before cleaning: {len(df)} rows")
df.dropna(inplace=True)
print(f"After dropping NAs: {len(df)} rows")

# --- Encode categorical variables ---
# Month -> ordinal encoding
month_map = {'Jan':1, 'Feb':2, 'Mar':3, 'Apr':4, 'May':5, 'June':6,
             'Jul':7, 'Aug':8, 'Sep':9, 'Oct':10, 'Nov':11, 'Dec':12}
df['Month'] = df['Month'].map(month_map)

# VisitorType -> label encoding
le_visitor = LabelEncoder()
df['VisitorType'] = le_visitor.fit_transform(df['VisitorType'])
print(f"VisitorType classes: {list(le_visitor.classes_)}")

# Boolean -> int
df['Weekend'] = df['Weekend'].astype(int)
df['Revenue'] = df['Revenue'].astype(int)

# --- Separate features and target ---
X = df.drop('Revenue', axis=1)
y = df['Revenue']

# --- Feature scaling ---
scaler = StandardScaler()
numeric_cols = X.select_dtypes(include=[np.number]).columns
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

print(f"\nFeatures: {X.shape[1]} | Samples: {X.shape[0]}")
print(f"Class distribution: {y.value_counts().to_dict()}")

# --- Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

In [ ]:
# ----- Correlation Heatmap -----
plt.figure(figsize=(14, 10))
correlation = df.corr()
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.5,
            square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('models/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Correlation heatmap saved")

---

# Task 2: Experiment Tracking using MLflow (4 Marks)

We train **4 models** and track all parameters, metrics, and artifacts in MLflow:
1. Logistic Regression
2. Random Forest
3. XGBoost
4. Gradient Boosting

In [ ]:
# ======================== MLFLOW EXPERIMENT TRACKING ========================

# Set up MLflow
EXPERIMENT_NAME = "Online_Shoppers_Intention"
mlflow.set_experiment(EXPERIMENT_NAME)
os.makedirs('models', exist_ok=True)

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"\nNote: After running this notebook, launch the MLflow UI with:")
print(f"  mlflow ui --port 5000")
print(f"Then open http://localhost:5000 in your browser.")

In [ ]:
# ======================== STAGE 3 & 4: MODEL TRAINING & EVALUATION ========================

# Trusted types for MLflow skops serialization (needed for tree-based models)
SKOPS_TRUSTED = [
    'sklearn.tree._tree.Tree',
    'sklearn.ensemble._forest.RandomForestClassifier',
    'sklearn.ensemble._gb.GradientBoostingClassifier',
    'numpy.float64', 'numpy.int64',
]

# Define models
models = {
    'Logistic_Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'
    ),
    'Random_Forest': RandomForestClassifier(
        n_estimators=200, max_depth=15, random_state=42,
        class_weight='balanced', n_jobs=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=42, eval_metric='logloss',
        scale_pos_weight=5, use_label_encoder=False
    ),
    'Gradient_Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        random_state=42
    ),
}

# Train and log each model
results = {}

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        print(f"\n{'='*60}")
        print(f"  Training: {name}")
        print(f"{'='*60}")
        
        # Log parameters
        params = model.get_params()
        for p_name, p_val in params.items():
            try:
                mlflow.log_param(p_name, p_val)
            except:
                pass
        
        # Train
        model.fit(X_train, y_train)
        
        # --- Evaluate on TRAIN set ---
        y_train_pred = model.predict(X_train)
        train_acc = accuracy_score(y_train, y_train_pred)
        mlflow.log_metric('train_accuracy', train_acc)
        
        # --- Evaluate on TEST set ---
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        
        test_acc = accuracy_score(y_test, y_pred)
        test_prec = precision_score(y_test, y_pred, zero_division=0)
        test_rec = recall_score(y_test, y_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_pred, zero_division=0)
        test_auc = roc_auc_score(y_test, y_proba)
        
        # Log test metrics
        mlflow.log_metric('test_accuracy', test_acc)
        mlflow.log_metric('test_precision', test_prec)
        mlflow.log_metric('test_recall', test_rec)
        mlflow.log_metric('test_f1_score', test_f1)
        mlflow.log_metric('test_roc_auc', test_auc)
        
        # Log model (with skops trusted types for tree-based models)
        try:
            mlflow.sklearn.log_model(
                model, artifact_path=name,
                skops_trusted_types=SKOPS_TRUSTED
            )
        except Exception as e:
            print(f"  Note: MLflow model logging fallback - {e}")
            mlflow.sklearn.log_model(
                model, artifact_path=name,
                serialization_format='pickle'
            )
        
        # Save model locally
        model_path = f'models/{name}.joblib'
        joblib.dump(model, model_path)
        mlflow.log_artifact(model_path)
        
        # Tags
        mlflow.set_tag('model_type', name)
        mlflow.set_tag('dataset', 'Online_Shoppers_Intention')
        
        # Store results
        results[name] = {
            'train_accuracy': train_acc,
            'test_accuracy': test_acc,
            'precision': test_prec,
            'recall': test_rec,
            'f1_score': test_f1,
            'roc_auc': test_auc,
            'model': model,
        }
        
        # Print summary
        print(f"  Train Accuracy : {train_acc:.4f}")
        print(f"  Test Accuracy  : {test_acc:.4f}")
        print(f"  Precision      : {test_prec:.4f}")
        print(f"  Recall         : {test_rec:.4f}")
        print(f"  F1-Score       : {test_f1:.4f}")
        print(f"  ROC-AUC        : {test_auc:.4f}")
        print(f"  Logged to MLflow")

print(f"\n{'='*60}")
print("All models trained and logged to MLflow")
print(f"{'='*60}")

In [ ]:
# ======================== MODEL COMPARISON TABLE ========================

comparison_df = pd.DataFrame({
    name: {
        'Train Accuracy': r['train_accuracy'],
        'Test Accuracy': r['test_accuracy'],
        'Precision': r['precision'],
        'Recall': r['recall'],
        'F1-Score': r['f1_score'],
        'ROC-AUC': r['roc_auc'],
    }
    for name, r in results.items()
}).T

print("\nMODEL COMPARISON")
print("=" * 80)
print(comparison_df.round(4).to_string())

# Best model by F1
best_model_name = comparison_df['F1-Score'].idxmax()
best_f1 = comparison_df.loc[best_model_name, 'F1-Score']
print(f"\nBest Model (by F1-Score): {best_model_name} ({best_f1:.4f})")

In [ ]:
# ======================== VISUALIZATION: MODEL COMPARISON ========================

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Model Comparison - MLflow Experiment Results', fontsize=16, fontweight='bold')

metrics_to_plot = ['Test Accuracy', 'F1-Score', 'ROC-AUC']
colors_bar = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for idx, metric in enumerate(metrics_to_plot):
    bars = axes[idx].bar(comparison_df.index, comparison_df[metric],
                          color=colors_bar, edgecolor='white', linewidth=1.5)
    axes[idx].set_title(metric, fontsize=14, fontweight='bold')
    axes[idx].set_ylim(0, 1)
    axes[idx].tick_params(axis='x', rotation=30)
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        axes[idx].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{height:.3f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig('models/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Model comparison chart saved")

In [ ]:
# ======================== ROC CURVES ========================

plt.figure(figsize=(10, 8))
colors_roc = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for (name, r), color in zip(results.items(), colors_roc):
    model = r['model']
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val = r['roc_auc']
    plt.plot(fpr, tpr, color=color, lw=2.5, label=f'{name} (AUC={auc_val:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random (AUC=0.500)')
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.title('ROC Curves - All Models', fontsize=16, fontweight='bold')
plt.legend(fontsize=11, loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('models/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("ROC curves saved")

In [ ]:
# ======================== CONFUSION MATRICES ========================

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Confusion Matrices - All Models', fontsize=16, fontweight='bold')

for idx, (name, r) in enumerate(results.items()):
    model = r['model']
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    axes[idx].set_title(name.replace('_', ' '), fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('models/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrices saved")

In [ ]:
# ======================== CLASSIFICATION REPORTS ========================

for name, r in results.items():
    model = r['model']
    y_pred = model.predict(X_test)
    print(f"\n{'='*60}")
    print(f"Classification Report: {name}")
    print(f"{'='*60}")
    print(classification_report(y_test, y_pred, target_names=['No Purchase', 'Purchase']))

In [ ]:
# ======================== FEATURE IMPORTANCE (Random Forest) ========================

rf_model = results['Random_Forest']['model']
feature_imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
feature_imp.plot.barh(color='#4CAF50', edgecolor='white')
plt.title('Feature Importance - Random Forest', fontsize=16, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('models/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Feature importance chart saved")

---

# Task 3: Dataset Versioning with DVC (4 Marks)

DVC (Data Version Control) tracks large datasets alongside Git.

### Steps to set up DVC (run in terminal):

```bash
# 1. Initialize DVC
dvc init

# 2. Add original dataset (Version 1)
dvc add Dataset/online_shoppers_intention.csv
git add Dataset/online_shoppers_intention.csv.dvc Dataset/.gitignore .dvc/.gitignore .dvc/config
git commit -m "DVC: Add original dataset (Version 1)"
git tag v1.0-original-dataset

# 3. Create modified dataset (Version 2 - with feature engineering)
# Run the cell below to generate the modified dataset, then:
dvc add Dataset/online_shoppers_intention.csv
git add Dataset/online_shoppers_intention.csv.dvc
git commit -m "DVC: Modified dataset with feature engineering (Version 2)"
git tag v2.0-modified-dataset

# 4. Compare versions
dvc diff v1.0-original-dataset v2.0-modified-dataset

# 5. Switch between versions
git checkout v1.0-original-dataset -- Dataset/online_shoppers_intention.csv.dvc
dvc checkout
```

In [ ]:
# ======================== DVC SETUP (Programmatic) ========================
# Run DVC commands from within the notebook

import subprocess

def run_cmd(cmd, show_output=True):
    """Run a shell command and return output."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = result.stdout + result.stderr
    if show_output:
        print(f"$ {cmd}")
        print(output)
    return result.returncode, output

if not IN_COLAB:
    # Initialize DVC (idempotent)
    run_cmd('dvc init --no-scm')
    print("DVC initialized")
else:
    print("DVC operations are best run locally. See instructions above.")

In [ ]:
# ======================== VERSION 1: Original Dataset ========================

print("VERSION 1: Original Dataset")
print("=" * 60)

v1 = pd.read_csv(DATA_PATH)
print(f"Shape: {v1.shape}")
print(f"Columns: {v1.columns.tolist()}")
print(f"\nSample:")
v1.head(3)

In [ ]:
# ======================== VERSION 2: Modified Dataset (Feature Engineering) ========================

print("VERSION 2: Modified Dataset with Feature Engineering")
print("=" * 60)

v2 = df_raw.copy()

# Add engineered features
v2['TotalPages'] = v2['Administrative'] + v2['Informational'] + v2['ProductRelated']
v2['TotalDuration'] = (v2['Administrative_Duration'] +
                       v2['Informational_Duration'] +
                       v2['ProductRelated_Duration'])
v2['BounceExitRatio'] = v2['BounceRates'] / (v2['ExitRates'] + 1e-6)

print(f"Shape: {v2.shape}")
print(f"New columns added: TotalPages, TotalDuration, BounceExitRatio")
print(f"\nSample:")
v2[['TotalPages', 'TotalDuration', 'BounceExitRatio', 'Revenue']].head(5)

In [ ]:
# Save modified dataset (this would be Version 2 for DVC)
v2_path = 'Dataset/online_shoppers_intention_v2.csv'
v2.to_csv(v2_path, index=False)
print(f"Modified dataset saved to: {v2_path}")

# DVC commands to track (run in terminal)
if not IN_COLAB:
    run_cmd(f'dvc add {v2_path}')
    print("\nModified dataset added to DVC tracking")

In [ ]:
# ======================== VERSION COMPARISON ========================

print("DATASET VERSION COMPARISON")
print("=" * 60)

comparison = {
    'Metric': ['Rows', 'Columns', 'Features', 'New Features', 'File Size (KB)'],
    'Version 1 (Original)': [
        v1.shape[0], v1.shape[1], v1.shape[1] - 1, 0,
        round(os.path.getsize(DATA_PATH) / 1024, 2)
    ],
    'Version 2 (Modified)': [
        v2.shape[0], v2.shape[1], v2.shape[1] - 1, 3,
        round(os.path.getsize(v2_path) / 1024, 2) if os.path.exists(v2_path) else 'N/A'
    ],
}

version_comparison_df = pd.DataFrame(comparison)
print(version_comparison_df.to_string(index=False))

print("\nBenefits of Data Versioning with DVC:")
print("   1. Reproducibility: Easily reproduce experiments with exact dataset version")
print("   2. Collaboration: Team members can switch between dataset versions seamlessly")
print("   3. Traceability: Every dataset change is tracked alongside code changes in Git")
print("   4. Storage efficiency: DVC deduplicates data, saving storage space")
print("   5. Rollback: Quickly revert to previous dataset versions if issues arise")

---

# Task 4: Workflow Automation Design using Kubeflow (4 Marks)

We simulate a Kubeflow pipeline with 6 steps that form a DAG:

```
Data Collection -> Data Validation -> Preprocessing -> Training -> Evaluation -> Deployment
```

### How Automation Improves Reproducibility and Scalability

| Aspect | Without Automation | With Kubeflow Pipeline |
|--------|-------------------|------------------------|
| **Reproducibility** | Manual steps, undocumented order | Every step is codified, versioned, and deterministic |
| **Scalability** | Single machine, sequential | Distributed execution across containers |
| **Error Handling** | Ad-hoc, easy to miss | Built-in retry, logging, and alerting |
| **Collaboration** | Works on my machine problem | Shared pipeline definitions, consistent environments |
| **Auditing** | No history | Full lineage of every run with inputs/outputs |

In [ ]:
# ======================== SIMULATED KUBEFLOW PIPELINE ========================

import time

def pipeline_step(step_num, name, description, func=None):
    """Execute a pipeline step with logging."""
    print(f"\n{'='*60}")
    print(f"  STEP {step_num}: {name}")
    print(f"  {description}")
    print(f"{'='*60}")
    start = time.time()
    
    result = {}
    if func:
        result = func()
    
    elapsed = round(time.time() - start, 3)
    print(f"  Completed in {elapsed}s")
    result['duration_sec'] = elapsed
    result['timestamp'] = datetime.now().isoformat()
    return result

# ---- Define pipeline steps ----

def step_data_collection():
    df = pd.read_csv(DATA_PATH)
    print(f"  Loaded {df.shape[0]} rows, {df.shape[1]} columns")
    return {'rows': df.shape[0], 'columns': df.shape[1], 'status': 'SUCCESS'}

def step_data_validation():
    df = pd.read_csv(DATA_PATH)
    missing = int(df.isnull().sum().sum())
    dupes = int(df.duplicated().sum())
    schema_ok = len(df.columns) >= 18
    print(f"  Schema valid: {schema_ok}")
    print(f"  Missing values: {missing}")
    print(f"  Duplicates: {dupes}")
    return {'schema_valid': schema_ok, 'missing': missing, 'duplicates': dupes, 'status': 'SUCCESS'}

def step_preprocessing():
    print(f"  Features: {X_train.shape[1]}")
    print(f"  Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
    return {'features': X_train.shape[1], 'train_size': X_train.shape[0], 'status': 'SUCCESS'}

def step_training():
    print(f"  Models trained: {list(results.keys())}")
    print(f"  Tracking: MLflow")
    return {'models': list(results.keys()), 'tracking': 'MLflow', 'status': 'SUCCESS'}

def step_evaluation():
    best = max(results, key=lambda k: results[k]['f1_score'])
    print(f"  Best model: {best} (F1={results[best]['f1_score']:.4f})")
    return {'best_model': best, 'best_f1': results[best]['f1_score'], 'status': 'SUCCESS'}

def step_deployment():
    print(f"  Target: Google Vertex AI")
    print(f"  Format: joblib / MLflow Model")
    print(f"  Serving: REST API Endpoint")
    return {'target': 'Vertex AI', 'format': 'joblib', 'status': 'SUCCESS'}

# ---- Run the pipeline ----
print("\n" + "= " * 30)
print("  KUBEFLOW PIPELINE - SIMULATED EXECUTION")
print("= " * 30)

pipeline_results = {}
steps = [
    (1, 'DATA COLLECTION', 'Load raw CSV data from source', step_data_collection),
    (2, 'DATA VALIDATION', 'Validate schema, missing values, duplicates', step_data_validation),
    (3, 'DATA PREPROCESSING', 'Encode, scale, split train/test', step_preprocessing),
    (4, 'MODEL TRAINING', 'Train 4 models with MLflow tracking', step_training),
    (5, 'MODEL EVALUATION', 'Compare models, select best', step_evaluation),
    (6, 'DEPLOYMENT', 'Deploy to Google Vertex AI', step_deployment),
]

for num, name, desc, func in steps:
    pipeline_results[name] = pipeline_step(num, name, desc, func)

# Pipeline summary
print(f"\n{'='*60}")
print("  PIPELINE EXECUTION SUMMARY")
print(f"{'='*60}")
total_time = sum(r['duration_sec'] for r in pipeline_results.values())
for name, r in pipeline_results.items():
    status = r.get('status', 'DONE')
    print(f"  {status} | {name} ({r['duration_sec']}s)")
print(f"\n  Total pipeline time: {total_time:.3f}s")
print(f"  All steps completed successfully")

In [ ]:
# ======================== PIPELINE DAG VISUALIZATION ========================

fig, ax = plt.subplots(figsize=(16, 5))
ax.set_xlim(0, 7)
ax.set_ylim(0, 2)
ax.set_aspect('equal')
ax.axis('off')
fig.suptitle('Kubeflow Pipeline DAG - Online Shoppers Intention', fontsize=16, fontweight='bold')

step_names = ['Data\nCollection', 'Data\nValidation', 'Preprocessing',
              'Model\nTraining', 'Model\nEvaluation', 'Deployment']
step_colors = ['#2196F3', '#03A9F4', '#00BCD4', '#4CAF50', '#FF9800', '#F44336']

for i, (name, color) in enumerate(zip(step_names, step_colors)):
    x = i + 0.5
    # Draw box
    rect = plt.Rectangle((x - 0.4, 0.6), 0.8, 0.8, linewidth=2,
                          edgecolor=color, facecolor=color, alpha=0.8,
                          zorder=2, clip_on=False)
    ax.add_patch(rect)
    ax.text(x, 1.0, name, ha='center', va='center', fontsize=10,
            fontweight='bold', color='white', zorder=3)
    # Draw arrow
    if i < len(step_names) - 1:
        ax.annotate('', xy=(x + 0.6, 1.0), xytext=(x + 0.4, 1.0),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=2.5),
                    zorder=1)

plt.tight_layout()
plt.savefig('models/pipeline_dag.png', dpi=150, bbox_inches='tight')
plt.show()
print("Pipeline DAG visualization saved")

---

# Task 5: Feature Store and Cloud Platform Study (4 Marks)

## Part A: Feast Feature Store

### Three Key Features Managed by Feast

| Feature | Type | Description | Why Important |
|---------|------|-------------|---------------|
| **PageValues** | Float64 | Avg value of pages visited before transaction | Strongest predictor of purchase intent |
| **BounceRates** | Float64 | % of visitors who leave without further requests | Indicates session engagement quality |
| **ProductRelated_Duration** | Float64 | Time spent on product pages | Direct signal of purchase interest |

### How Feast Ensures Consistency

```
                    +-------------------+
                    |  Feature Store    |
                    |    (Feast)        |
                    +---------+---------+
                              |
              +---------------+---------------+
              v               v               v
    +---------------+  +--------------+  +----------------+
    |  Offline      |  |  Online      |  |  Feature       |
    |  Store        |  |  Store       |  |  Registry      |
    | (Parquet/BQ)  |  | (Redis/DB)   |  | (Metadata)     |
    +-------+-------+  +------+-------+  +----------------+
            |                 |
     Training Data     Real-time
     (Batch)           Inference
```

**Key Benefits:**
1. **No train-serve skew** - Same feature definitions for training and inference
2. **Point-in-time correctness** - Historical features joined correctly to prevent data leakage
3. **Feature reuse** - Multiple models can share the same feature definitions
4. **Low-latency serving** - Online store enables real-time predictions

In [ ]:
# ======================== FEAST FEATURE STORE DEMONSTRATION ========================

print("""
================================================================
       FEAST FEATURE STORE - Conceptual Overview
================================================================

THREE KEY FEATURES MANAGED BY FEAST:

  1. PageValues (Float64)
     -> Average value of pages visited before completing a
        transaction. Critical predictor of purchase intent.
     -> Feast ensures same transformation pipeline serves both
        batch training and real-time inference.

  2. BounceRates (Float64)
     -> Percentage of visitors who enter and leave from that
        page without triggering other requests.
     -> Feast provides point-in-time correct joins to prevent
        data leakage between train/test.

  3. ProductRelated_Duration (Float64)
     -> Total time spent on product-related pages.
     -> Feast's online store enables low-latency feature
        retrieval for real-time predictions.

================================================================
HOW FEAST ENSURES CONSISTENCY:

  Training:
    -> get_historical_features() fetches point-in-time correct
       feature values from the offline store (e.g., Parquet/BQ).

  Inference:
    -> get_online_features() retrieves the latest feature values
       from the online store (e.g., Redis/SQLite) with low
       latency for real-time serving.

  Benefit:
    -> Single source of truth for feature definitions
    -> No train-serve skew
    -> Feature reuse across multiple models
    -> Versioned feature transformations
================================================================
""")

In [ ]:
# ======================== FEAST FEATURE DEFINITIONS (Conceptual Code) ========================

print("Feast Feature View Definitions (Conceptual)")
print("=" * 60)

feast_code = '''
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float64, Int64, String

# Entity definition
shopper_session = Entity(
    name="session_id",
    description="Unique browsing session identifier",
)

# Data Source
session_source = FileSource(
    path="feast_feature_store/data/sessions.parquet",
    timestamp_field="event_timestamp",
)

# Feature View 1: Page Features
session_page_features = FeatureView(
    name="session_page_features",
    entities=[shopper_session],
    ttl=timedelta(days=365),
    schema=[
        Field(name="Administrative", dtype=Int64),
        Field(name="Informational", dtype=Int64),
        Field(name="ProductRelated", dtype=Int64),
    ],
    source=session_source,
)

# Feature View 2: Behaviour Features
session_behaviour_features = FeatureView(
    name="session_behaviour_features",
    entities=[shopper_session],
    ttl=timedelta(days=365),
    schema=[
        Field(name="BounceRates", dtype=Float64),
        Field(name="ExitRates", dtype=Float64),
        Field(name="PageValues", dtype=Float64),
    ],
    source=session_source,
)

# Feature View 3: Context Features
session_context_features = FeatureView(
    name="session_context_features",
    entities=[shopper_session],
    ttl=timedelta(days=365),
    schema=[
        Field(name="OperatingSystems", dtype=Int64),
        Field(name="Browser", dtype=Int64),
        Field(name="Region", dtype=Int64),
    ],
    source=session_source,
)

# Usage in Training:
#   features = store.get_historical_features(
#       entity_df=entity_dataframe,
#       features=["session_behaviour_features:PageValues",
#                 "session_behaviour_features:BounceRates"]
#   )
#
# Usage in Inference:
#   features = store.get_online_features(
#       features=["session_behaviour_features:PageValues"],
#       entity_rows=[{"session_id": "abc123"}]
#   )
'''

print(feast_code)

## Part B: Cloud Platform Comparison

### Google Cloud Vertex AI vs AWS SageMaker

| Feature | Google Vertex AI | AWS SageMaker |
|---------|-----------------|---------------|
| **Services Offered** | Unified ML platform: training, tuning, deployment, pipelines, feature store | End-to-end ML: Studio, training, hosting, Ground Truth, Clarify |
| **Experiment Tracking** | Vertex AI Experiments (built-in), MLflow integration | SageMaker Experiments (built-in), MLflow integration |
| **AutoML Support** | Vertex AI AutoML (tabular, image, text, video) | SageMaker Autopilot (tabular), JumpStart (pre-trained) |
| **Deployment Options** | Online/Batch prediction, endpoints, A/B testing, Knative | Real-time endpoints, batch transform, serverless, multi-model |
| **Cost Considerations** | Pay-per-use, preemptible VMs, sustained discounts | Pay-per-use, spot instances, savings plans |
| **Pipeline Tool** | Vertex AI Pipelines (Kubeflow-based) | SageMaker Pipelines (proprietary) |
| **Feature Store** | Vertex AI Feature Store | SageMaker Feature Store |
| **Monitoring** | Vertex AI Model Monitoring (drift detection) | SageMaker Model Monitor (data quality, bias) |
| **Integration** | BigQuery, Dataflow, TFX, Kubeflow | S3, Glue, Step Functions, Lambda |

### Recommendation

For this project, **Google Vertex AI** is chosen because:
1. Native Kubeflow Pipelines support aligns with our pipeline design
2. Seamless integration with BigQuery for large-scale feature engineering
3. Competitive pricing with sustained-use discounts
4. Built-in model monitoring with automatic drift detection

In [ ]:
# ======================== CLOUD PLATFORM COMPARISON VISUALIZATION ========================

categories = ['Services', 'Experiment\nTracking', 'AutoML', 'Deployment', 'Cost\nEfficiency', 'Monitoring']
vertex_scores = [9, 8, 9, 8, 8, 9]
sagemaker_scores = [9, 8, 8, 9, 7, 8]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, vertex_scores, width, label='Google Vertex AI',
               color='#4285F4', edgecolor='white', linewidth=1.5)
bars2 = ax.bar(x + width/2, sagemaker_scores, width, label='AWS SageMaker',
               color='#FF9900', edgecolor='white', linewidth=1.5)

ax.set_ylabel('Score (out of 10)', fontsize=12)
ax.set_title('Cloud Platform Comparison: Vertex AI vs SageMaker', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11)
ax.legend(fontsize=12)
ax.set_ylim(0, 11)
ax.grid(axis='y', alpha=0.3)

# Value labels
for bar in bars1 + bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
            f'{height}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('models/cloud_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Cloud comparison chart saved")

---

# Model Monitoring (Bonus)

Simulate production monitoring of the deployed model:

```
Model deployed
      |
      v
Predictions generated
      |
      v
Monitor:
  - Prediction behaviour
  - Model performance drift
  - Data distribution changes
  - Error rates
```

In [ ]:
# ======================== MODEL MONITORING SIMULATION ========================

# Use the best model
best_name = max(results, key=lambda k: results[k]['f1_score'])
best_model = results[best_name]['model']
baseline_metrics = {
    'accuracy': results[best_name]['test_accuracy'],
    'precision': results[best_name]['precision'],
    'recall': results[best_name]['recall'],
    'f1_score': results[best_name]['f1_score'],
}

print(f"Monitoring Model: {best_name}")
print(f"Baseline Metrics: {baseline_metrics}")

In [ ]:
# ---- 1. Prediction Behaviour Monitoring ----

print("\n" + "="*70)
print("  PREDICTION BEHAVIOUR MONITORING")
print("="*70)

y_pred_test = best_model.predict(X_test)
y_proba_test = best_model.predict_proba(X_test)[:, 1]

print(f"\n  Total predictions: {len(y_pred_test)}")
print(f"  Positive (Purchase): {y_pred_test.sum()} ({y_pred_test.mean():.2%})")
print(f"  Negative (No Purchase): {(y_pred_test==0).sum()} ({(y_pred_test==0).mean():.2%})")
print(f"\n  Confidence Statistics:")
print(f"    Mean confidence:  {y_proba_test.mean():.4f}")
print(f"    Std confidence:   {y_proba_test.std():.4f}")
print(f"    Min confidence:   {y_proba_test.min():.4f}")
print(f"    Max confidence:   {y_proba_test.max():.4f}")
low_conf = ((y_proba_test > 0.4) & (y_proba_test < 0.6)).sum()
print(f"    Low-confidence predictions (0.4-0.6): {low_conf}")

# Prediction distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(y_proba_test, bins=50, color='#2196F3', edgecolor='white', alpha=0.8)
axes[0].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Threshold (0.5)')
axes[0].set_title('Prediction Probability Distribution', fontweight='bold')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Count')
axes[0].legend()

labels = ['No Purchase', 'Purchase']
sizes = [(y_pred_test==0).sum(), y_pred_test.sum()]
axes[1].pie(sizes, labels=labels, autopct='%1.1f%%', colors=['#2196F3', '#FF5722'],
            startangle=90, explode=(0, 0.05))
axes[1].set_title('Prediction Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig('models/monitoring_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- 2. Model Performance Monitoring ----

print("\n" + "="*70)
print("  MODEL PERFORMANCE MONITORING")
print("="*70)

# Simulate multiple batches to show performance over time
np.random.seed(42)
n_batches = 5
batch_metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}

THRESHOLD = 0.05  # 5% drop triggers alert

for batch_i in range(n_batches):
    # Simulate batch with slight data shift
    noise = np.random.normal(0, 0.05 * batch_i, X_test.shape)  # Increasing noise
    X_batch = X_test + noise
    y_batch_pred = best_model.predict(X_batch)
    
    acc = accuracy_score(y_test, y_batch_pred)
    prec = precision_score(y_test, y_batch_pred, zero_division=0)
    rec = recall_score(y_test, y_batch_pred, zero_division=0)
    f1 = f1_score(y_test, y_batch_pred, zero_division=0)
    
    batch_metrics['accuracy'].append(acc)
    batch_metrics['precision'].append(prec)
    batch_metrics['recall'].append(rec)
    batch_metrics['f1_score'].append(f1)
    
    delta = acc - baseline_metrics['accuracy']
    alert = "DRIFT DETECTED" if delta < -THRESHOLD else "OK"
    print(f"  Batch {batch_i+1}: acc={acc:.4f} (delta={delta:+.4f}) {alert}")

# Plot performance over batches
fig, ax = plt.subplots(figsize=(12, 6))
for metric_name, values in batch_metrics.items():
    ax.plot(range(1, n_batches+1), values, marker='o', linewidth=2.5, label=metric_name)
    ax.axhline(y=baseline_metrics[metric_name], linestyle='--', alpha=0.4)

ax.set_xlabel('Batch Number', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Performance Monitoring Over Batches (Simulated Drift)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('models/monitoring_performance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- 3. Data Drift Monitoring ----

print("\n" + "="*70)
print("  DATA DRIFT MONITORING")
print("="*70)

# Simulate drifted data
X_drifted = X_test.copy()
# Add drift to some features
X_drifted['PageValues'] = X_drifted['PageValues'] + np.random.normal(2, 1, len(X_drifted))
X_drifted['BounceRates'] = X_drifted['BounceRates'] + np.random.normal(0.5, 0.3, len(X_drifted))

numeric_cols_list = X_test.select_dtypes(include=[np.number]).columns.tolist()
drift_results = {}

for col in numeric_cols_list:
    ref_mean = X_train[col].mean()
    cur_mean = X_drifted[col].mean()
    ref_std = X_train[col].std()
    shift = abs(cur_mean - ref_mean) / (ref_std + 1e-8)
    drifted = shift > 1.0
    drift_results[col] = {'shift': shift, 'drifted': drifted}
    if drifted:
        print(f"  WARNING {col}: normalized shift = {shift:.4f} - DRIFT DETECTED")

stable_count = sum(1 for v in drift_results.values() if not v['drifted'])
drift_count = sum(1 for v in drift_results.values() if v['drifted'])
print(f"\n  Summary: {drift_count} features drifted, {stable_count} stable")

# Visualize drift
drift_df = pd.DataFrame({
    'Feature': drift_results.keys(),
    'Shift': [v['shift'] for v in drift_results.values()],
    'Drifted': [v['drifted'] for v in drift_results.values()],
})
drift_df = drift_df.sort_values('Shift', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
colors_drift = ['#F44336' if d else '#4CAF50' for d in drift_df['Drifted']]
ax.barh(drift_df['Feature'], drift_df['Shift'], color=colors_drift, edgecolor='white')
ax.axvline(x=1.0, color='red', linestyle='--', linewidth=2, label='Drift Threshold')
ax.set_xlabel('Normalized Mean Shift', fontsize=12)
ax.set_title('Data Drift Detection (Reference vs Current)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('models/monitoring_data_drift.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- 4. Error Monitoring ----

print("\n" + "="*70)
print("  ERROR MONITORING")
print("="*70)

# Simulate a batch with some corrupted rows
X_corrupted = X_test.copy()
corrupt_indices = np.random.choice(len(X_corrupted), size=5, replace=False)
X_corrupted.iloc[corrupt_indices] = np.nan

errors = []
for i in range(len(X_corrupted)):
    try:
        row = X_corrupted.iloc[[i]]
        if row.isnull().any().any():
            raise ValueError(f"NaN values detected in row {i}")
        best_model.predict(row)
    except Exception as e:
        errors.append({'index': i, 'error': str(e)})

print(f"  Total predictions attempted: {len(X_corrupted)}")
print(f"  Successful: {len(X_corrupted) - len(errors)}")
print(f"  Failed: {len(errors)}")
print(f"  Error rate: {len(errors)/len(X_corrupted):.2%}")
if errors:
    print(f"\n  Sample errors:")
    for err in errors[:3]:
        print(f"    Row {err['index']}: {err['error']}")

In [ ]:
# ======================== MONITORING SUMMARY DASHBOARD ========================

print("\n" + "="*70)
print("              MONITORING SUMMARY DASHBOARD")
print("="*70)
print(f"""\n
  Model: {best_name}
  Status: DEPLOYED

  Predictions: {len(y_pred_test)} total
     Positive rate: {y_pred_test.mean():.2%}
     Avg confidence: {y_proba_test.mean():.4f}

  Performance:
     Accuracy:  {baseline_metrics['accuracy']:.4f}
     F1-Score:  {baseline_metrics['f1_score']:.4f}
     Status:    No significant drift

  Data Drift: {drift_count} feature(s) drifted
     Features monitored: {len(numeric_cols_list)}

  Errors: {len(errors)} / {len(X_corrupted)} ({len(errors)/len(X_corrupted):.2%} error rate)
""")

---

# Cloud Deployment Guide - Google Vertex AI

### Step-by-Step Deployment Instructions

#### Prerequisites
```bash
pip install google-cloud-aiplatform
gcloud auth login
gcloud config set project YOUR_PROJECT_ID
```

#### Step 1: Save Model Artifact
```python
import joblib
joblib.dump(best_model, 'models/best_model.joblib')
```

#### Step 2: Upload Model to Vertex AI
```python
from google.cloud import aiplatform

aiplatform.init(
    project='YOUR_PROJECT_ID',
    location='us-central1',
)

model = aiplatform.Model.upload(
    display_name='online-shoppers-intention-model',
    artifact_uri='gs://YOUR_BUCKET/models/',
    serving_container_image_uri='us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest',
)
```

#### Step 3: Deploy to Endpoint
```python
endpoint = model.deploy(
    deployed_model_display_name='shoppers-model-v1',
    machine_type='n1-standard-2',
    min_replica_count=1,
    max_replica_count=3,
)
```

#### Step 4: Make Predictions
```python
prediction = endpoint.predict(
    instances=[
        [0, 0.0, 0, 0.0, 1, 0.0, 0.2, 0.1, 0.0, 0.0, 2, 1, 1, 1, 1, 1, 0]
    ]
)
print(f"Prediction: {prediction.predictions}")
```

#### Step 5: Enable Monitoring
```python
from google.cloud.aiplatform import model_monitoring

job = aiplatform.ModelDeploymentMonitoringJob.create(
    display_name='shoppers-monitoring',
    endpoint=endpoint,
    logging_sampling_strategy=model_monitoring.SamplingStrategy(
        random_sample_config=model_monitoring.SamplingStrategy.RandomSampleConfig(
            sample_rate=0.8
        )
    ),
    schedule_config=model_monitoring.ScheduleConfig(
        monitor_interval={'seconds': 3600}
    ),
)
```

In [ ]:
# ======================== SAVE BEST MODEL ========================

# Save the best model for deployment
best_model_path = f'models/best_model_{best_name}.joblib'
joblib.dump(best_model, best_model_path)
print(f"Best model saved to: {best_model_path}")

# Save the scaler and encoder for inference
joblib.dump(scaler, 'models/scaler.joblib')
joblib.dump(le_visitor, 'models/label_encoder_visitor.joblib')
print("Scaler and encoder saved")

# Summary
print(f"\n{'='*60}")
print(f"  FINAL SUMMARY")
print(f"{'='*60}")
print(f"  Best Model: {best_name}")
print(f"  Test Accuracy: {results[best_name]['test_accuracy']:.4f}")
print(f"  Test F1-Score: {results[best_name]['f1_score']:.4f}")
print(f"  Test ROC-AUC:  {results[best_name]['roc_auc']:.4f}")
print(f"  Model saved:   {best_model_path}")
print(f"\n  To view MLflow experiments:")
print(f"    mlflow ui --port 5000")
print(f"  Then open: http://localhost:5000")

---

# Summary

| Task | Status | Tools Used |
|------|--------|------------|
| **Task 1:** MLOps Lifecycle Analysis | Complete | Diagram + Code |
| **Task 2:** Experiment Tracking | Complete | MLflow (4 models) |
| **Task 3:** Dataset Versioning | Complete | DVC + Git |
| **Task 4:** Workflow Automation | Complete | Kubeflow (Simulated) |
| **Task 5A:** Feature Store | Complete | Feast (Conceptual) |
| **Task 5B:** Cloud Comparison | Complete | Vertex AI vs SageMaker |
| **Bonus:** Monitoring | Complete | Custom monitoring module |
| **Bonus:** Cloud Deployment | Guide provided | Google Vertex AI |

### Repository Structure
```
MLOps-CIA1/
  MLOps_CIA1.ipynb            # Main notebook (this file)
  Dataset/
    online_shoppers_intention.csv       # Original dataset (DVC tracked)
    online_shoppers_intention_v2.csv    # Modified dataset (DVC tracked)
  src/
    data_preprocessing.py   # Data loading and preprocessing
    model_training.py       # Model training with MLflow
    monitoring.py           # Production monitoring
  pipeline/
    kubeflow_pipeline.py    # Simulated Kubeflow pipeline
  feast_feature_store/
    feature_definitions.py  # Feast feature definitions
  models/                   # Saved models and plots
  .dvc/                     # DVC configuration
  .gitignore
  requirements.txt
  README.md
```